# Notebook 02: Data Curation and Feature Engineering
## Transformation, Imputation, and Feature Derivation
**Author:** Data Engineering & Analytics Team

### Objective:
Execute attribute-specific data quality treatments without dropping useful fact records. Engineer all temporal, contextual, spatial, and categorization features required for dimensional loading and predictive analytics.

In [1]:
import os
import pandas as pd
import numpy as np

raw_path = '../data/raw/crimes_2024_2025.csv'
df = pd.read_csv(raw_path, low_memory=False)
initial_rows = len(df)
print(f'Initial Raw Record Count: {initial_rows:,}')

Initial Raw Record Count: 497,919


### 1. Attribute-Specific Treatments (No Blanket Deletions)

In [2]:
# Impute missing location description with UNKNOWN
df['location_description'] = df['location_description'].fillna('UNKNOWN').astype(str).str.strip().str.upper()
# Impute missing administrative Ward and Community Area with sentinel -1
df['ward'] = df['ward'].fillna(-1).astype(int)
df['community_area'] = df['community_area'].fillna(-1).astype(int)
# Geocoordinate flag
df['has_valid_coordinates'] = (df['latitude'].notnull() & df['longitude'].notnull()).astype(int)
print('Data quality treatments applied successfully.')

Data quality treatments applied successfully.


### 2. Standardize Offense Codes and Text Formatting

In [3]:
df['iucr'] = df['iucr'].astype(str).str.strip().str.zfill(4)
df['fbi_code'] = df['fbi_code'].astype(str).str.strip()
df['primary_type'] = df['primary_type'].astype(str).str.strip().str.upper()
df['district'] = df['district'].astype(str).str.strip().str.zfill(3)
df['beat'] = df['beat'].astype(str).str.strip().str.zfill(4)

### 3. Date & Time Decomposition

In [4]:
df['incident_datetime'] = pd.to_datetime(df['date'])
df['full_date'] = df['incident_datetime'].dt.date
df['year'] = df['incident_datetime'].dt.year
df['month'] = df['incident_datetime'].dt.month
df['quarter'] = df['incident_datetime'].dt.quarter
df['day'] = df['incident_datetime'].dt.day
df['day_of_week'] = df['incident_datetime'].dt.day_of_week + 1 # 1=Mon, 7=Sun
df['day_name'] = df['incident_datetime'].dt.day_name()
df['hour'] = df['incident_datetime'].dt.hour
df['minute'] = df['incident_datetime'].dt.minute
df['is_weekend'] = df['day_of_week'].apply(lambda d: 1 if d in [6, 7] else 0)

### 4. Operational TimeBands and Night Incident Flag
- **Night:** 21:00 to 04:59 (Hours 21-23, 0-4)
- **Morning:** 05:00 to 11:59 (Hours 5-11)
- **Afternoon:** 12:00 to 16:59 (Hours 12-16)
- **Evening:** 17:00 to 20:59 (Hours 17-20)

In [5]:
def get_time_band(h):
    if h in [21, 22, 23, 0, 1, 2, 3, 4]: return 'Night'
    elif h in [5, 6, 7, 8, 9, 10, 11]: return 'Morning'
    elif h in [12, 13, 14, 15, 16]: return 'Afternoon'
    else: return 'Evening'

df['time_band'] = df['hour'].apply(get_time_band)
df['is_night_incident'] = (df['time_band'] == 'Night').astype(int)
df['arrest_flag'] = (df['arrest'] == True).astype(int)
df['domestic_flag'] = (df['domestic'] == True).astype(int)

### 5. Crime and Location Operational Groupings

In [6]:
def categorize_crime(pt):
    pt = str(pt).strip().upper()
    violent = {'HOMICIDE', 'CRIMINAL SEXUAL ASSAULT', 'ROBBERY', 'ASSAULT', 'BATTERY', 'KIDNAPPING', 'INTIMIDATION', 'STALKING'}
    prop = {'THEFT', 'BURGLARY', 'MOTOR VEHICLE THEFT', 'ARSON', 'DECEPTIVE PRACTICE', 'CRIMINAL DAMAGE', 'CRIMINAL TRESPASS'}
    subst = {'NARCOTICS', 'OTHER NARCOTIC VIOLATION', 'LIQUOR LAW VIOLATION'}
    pub = {'WEAPONS VIOLATION', 'PUBLIC PEACE VIOLATION', 'INTERFERENCE WITH PUBLIC OFFICER', 'CONCEALED CARRY LICENSE VIOLATION', 'PROSTITUTION', 'GAMBLING', 'PUBLIC INDECENCY', 'OBSCENITY'}
    if pt in violent: return 'Violent Crime'
    elif pt in prop: return 'Property Crime'
    elif pt in subst: return 'Substance Offense'
    elif pt in pub: return 'Public Order & Weapons'
    else: return 'Other Offenses'

df['crime_category_group'] = df['primary_type'].apply(categorize_crime)
print(df['crime_category_group'].value_counts())

crime_category_group
Property Crime            270758
Violent Crime             155094
Other Offenses             40176
Public Order & Weapons     18074
Substance Offense          13817
Name: count, dtype: int64


### 6. Curation Reconciliation Summary

In [7]:
print(f'Initial Raw Rows: {initial_rows:,}')
print(f'Curated Rows:     {len(df):,}')
print(f'Preservation Rate: {len(df)/initial_rows*100:.2f}% (0 rows dropped)')
print(f'Total Arrests:    {df["arrest_flag"].sum():,}')
print(f'Total Domestic:   {df["domestic_flag"].sum():,}')

Initial Raw Rows: 497,919
Curated Rows:     497,919
Preservation Rate: 100.00% (0 rows dropped)
Total Arrests:    74,379
Total Domestic:   93,093
